


(a) Apply OrdinalEncoder to Satisfaction using the order Poor < Average < Good < Excellent. Encode Membership using a separately defined order. Display the encoded DataFrame and verify programmatically that the assigned values preserve the specified rankings.



In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    "Customer_ID": range(101, 113),
    "Satisfaction": [
        "Poor", "Good", "Average", "Excellent",
        "Poor", "Good", "Average", "Excellent",
        "Good", "Average", "Excellent", "Poor"
    ],
    "Purchase_Amount": [
        500, 1200, 800, 2500, 400, 1500,
        900, 3000, 1800, 700, 2800, 350
    ],
    "Membership": [
        "Bronze", "Silver", "Bronze", "Gold",
        "Bronze", "Gold", "Silver", "Gold",
        "Silver", "Bronze", "Gold", "Silver"
    ]
})

sat_order = ["Poor", "Average", "Good", "Excellent"]
mem_order = ["Bronze", "Silver", "Gold"]

encoder = OrdinalEncoder(categories=[sat_order, mem_order])

df[["Satisfaction", "Membership"]] = encoder.fit_transform(
    df[["Satisfaction", "Membership"]]
)

print(df)

print("\nSatisfaction ranking:")
print(df["Satisfaction"].unique())

print("\nMembership ranking:")
print(df["Membership"].unique())

assert df["Satisfaction"].between(0, 3).all()
assert df["Membership"].between(0, 2).all()

print("\nAll rankings verified!")


    Customer_ID  Satisfaction  Purchase_Amount  Membership
0           101           0.0              500         0.0
1           102           2.0             1200         1.0
2           103           1.0              800         0.0
3           104           3.0             2500         2.0
4           105           0.0              400         0.0
5           106           2.0             1500         2.0
6           107           1.0              900         1.0
7           108           3.0             3000         2.0
8           109           2.0             1800         1.0
9           110           1.0              700         0.0
10          111           3.0             2800         2.0
11          112           0.0              350         1.0

Satisfaction ranking:
[0. 2. 1. 3.]

Membership ranking:
[0. 1. 2.]

All rankings verified!


(b) Calculate the average purchase amount for each satisfaction category before and after encoding. Explain why the numerical encoding preserves category order but does not guarantee equal distances between satisfaction levels or establish a causal relationship with spending.

In [2]:
before = df.copy()

before["Satisfaction"] = encoder.inverse_transform(
    df[["Satisfaction", "Membership"]]
)[:, 0]

print("Average Purchase by Satisfaction (Before Encoding):")
print(before.groupby("Satisfaction")["Purchase_Amount"].mean())

print("\nAverage Purchase by Satisfaction (After Encoding):")
print(df.groupby("Satisfaction")["Purchase_Amount"].mean())

print("\nEncoded Satisfaction Mapping:")
for i, category in enumerate(sat_order):
    print(category, "=", i)


Average Purchase by Satisfaction (Before Encoding):
Satisfaction
Average       800.000000
Excellent    2766.666667
Good         1500.000000
Poor          416.666667
Name: Purchase_Amount, dtype: float64

Average Purchase by Satisfaction (After Encoding):
Satisfaction
0.0     416.666667
1.0     800.000000
2.0    1500.000000
3.0    2766.666667
Name: Purchase_Amount, dtype: float64

Encoded Satisfaction Mapping:
Poor = 0
Average = 1
Good = 2
Excellent = 3


2(a) Apply ordinal encoding to Study_Level using Low < Medium < High. Apply label encoding to the target Result, and demonstrate why assigning integer labels to the nominal feature City may introduce an artificial ordering. Display and compare the resulting representations.

In [3]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

df = pd.DataFrame({
    "Study_Level": [
        "Low", "Medium", "High", "Low", "High",
        "Medium", "High", "Low", "Medium", "High",
        "Low", "Medium"
    ],
    "City": [
        "Bhopal", "Delhi", "Mumbai", "Delhi",
        "Bhopal", "Mumbai", "Delhi", "Mumbai",
        "Bhopal", "Delhi", "Mumbai", "Bhopal"
    ],
    "Result": [
        "Fail", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Fail", "Pass"
    ]
})

print("Original Data:")
print(df)

ordinal = OrdinalEncoder(
    categories=[["Low", "Medium", "High"]]
)

df["Study_Level_Encoded"] = ordinal.fit_transform(
    df[["Study_Level"]]
).astype(int)

label = LabelEncoder()
df["Result_Encoded"] = label.fit_transform(df["Result"])

city_label = LabelEncoder()
df["City_Label_Encoded"] = city_label.fit_transform(df["City"])

print("\nEncoded Data:")
print(df)

print("\nCity Label Mapping:")
print(dict(zip(city_label.classes_,
               city_label.transform(city_label.classes_))))


Original Data:
   Study_Level    City Result
0          Low  Bhopal   Fail
1       Medium   Delhi   Pass
2         High  Mumbai   Pass
3          Low   Delhi   Fail
4         High  Bhopal   Pass
5       Medium  Mumbai   Pass
6         High   Delhi   Pass
7          Low  Mumbai   Fail
8       Medium  Bhopal   Pass
9         High   Delhi   Pass
10         Low  Mumbai   Fail
11      Medium  Bhopal   Pass

Encoded Data:
   Study_Level    City Result  Study_Level_Encoded  Result_Encoded  \
0          Low  Bhopal   Fail                    0               0   
1       Medium   Delhi   Pass                    1               1   
2         High  Mumbai   Pass                    2               1   
3          Low   Delhi   Fail                    0               0   
4         High  Bhopal   Pass                    2               1   
5       Medium  Mumbai   Pass                    1               1   
6         High   Delhi   Pass                    2               1   
7          Low  Mumb

(b) Build a preprocessing workflow that encodes the features and target appropriately, trains a classification model, and evaluates its predictions. Explain why the encoding strategy for a feature can differ from that for a target, and why arbitrary numerical labels for nominal features can affect certain algorithms.

In [4]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.DataFrame({
    "Study_Level": [
        "Low", "Medium", "High", "Low", "High",
        "Medium", "High", "Low", "Medium", "High",
        "Low", "Medium"
    ],
    "City": [
        "Bhopal", "Delhi", "Mumbai", "Delhi",
        "Bhopal", "Mumbai", "Delhi", "Mumbai",
        "Bhopal", "Delhi", "Mumbai", "Bhopal"
    ],
    "Result": [
        "Fail", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Fail", "Pass"
    ]
})

X = df[["Study_Level", "City"]]
y = LabelEncoder().fit_transform(df["Result"])

preprocessor = ColumnTransformer(
    transformers=[
        ("study", OrdinalEncoder(
            categories=[["Low", "Medium", "High"]]
        ), ["Study_Level"]),
        ("city", OneHotEncoder(handle_unknown="ignore"),
         ["City"])
    ]
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Actual Results:", y_test)
print("Predicted Results:", y_pred)
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Fail", "Pass"],
    zero_division=0
))


Actual Results: [0 1 1]
Predicted Results: [0 1 1]
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00         1
        Pass       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



3(a) Identify which columns are ordinal and nominal. Use OrdinalEncoder with explicit category orders for Size and Satisfaction. Demonstrate why Color should not be ordinal-encoded based on an invented order, and verify that all encoded columns contain the intended values.

In [5]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    "Size": [
        "Small", "Medium", "Large", "Medium",
        "Small", "Large", "XL", "Medium",
        "XL", "Small", "Large", "XL"
    ],
    "Color": [
        "Red", "Blue", "Green", "Red",
        "Green", "Blue", "Red", "Green",
        "Blue", "Red", "Blue", "Green"
    ],
    "Satisfaction": [
        "Poor", "Good", "Average", "Excellent",
        "Average", "Good", "Excellent", "Poor",
        "Good", "Average", "Excellent", "Good"
    ],
    "Order_Value": [
        500, 900, 1500, 1100, 650, 1700,
        2200, 700, 2100, 550, 1600, 2300
    ]
})

df_encoded = df.copy()

encoder = OrdinalEncoder(
    categories=[
        ["Small", "Medium", "Large", "XL"],
        ["Poor", "Average", "Good", "Excellent"]
    ]
)

df_encoded[["Size", "Satisfaction"]] = encoder.fit_transform(
    df[["Size", "Satisfaction"]]
).astype(int)

print("Ordinal Encoded Data:")
print(df_encoded)

print("\nSize Mapping:")
print(dict(zip(
    ["Small", "Medium", "Large", "XL"], [0, 1, 2, 3]
)))

print("\nSatisfaction Mapping:")
print(dict(zip(
    ["Poor", "Average", "Good", "Excellent"], [0, 1, 2, 3]
)))


Ordinal Encoded Data:
    Size  Color  Satisfaction  Order_Value
0      0    Red             0          500
1      1   Blue             2          900
2      2  Green             1         1500
3      1    Red             3         1100
4      0  Green             1          650
5      2   Blue             2         1700
6      3    Red             3         2200
7      1  Green             0          700
8      3   Blue             2         2100
9      0    Red             1          550
10     2   Blue             3         1600
11     3  Green             2         2300

Size Mapping:
{'Small': 0, 'Medium': 1, 'Large': 2, 'XL': 3}

Satisfaction Mapping:
{'Poor': 0, 'Average': 1, 'Good': 2, 'Excellent': 3}


(b) Build a ColumnTransformer that applies ordinal encoding to Size and Satisfaction and one-hot encoding to Color. Transform the dataset and explain why this combined preprocessing strategy is more appropriate than assigning arbitrary integer labels to every categorical feature.

In [6]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.DataFrame({
    "Size": [
        "Small", "Medium", "Large", "Medium",
        "Small", "Large", "XL", "Medium",
        "XL", "Small", "Large", "XL"
    ],
    "Color": [
        "Red", "Blue", "Green", "Red",
        "Green", "Blue", "Red", "Green",
        "Blue", "Red", "Blue", "Green"
    ],
    "Satisfaction": [
        "Poor", "Good", "Average", "Excellent",
        "Average", "Good", "Excellent", "Poor",
        "Good", "Average", "Excellent", "Good"
    ],
    "Order_Value": [
        500, 900, 1500, 1100, 650, 1700,
        2200, 700, 2100, 550, 1600, 2300
    ]
})

preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal", OrdinalEncoder(categories=[
            ["Small", "Medium", "Large", "XL"],
            ["Poor", "Average", "Good", "Excellent"]
        ]), ["Size", "Satisfaction"]),

        ("color", OneHotEncoder(
            sparse_output=False,
            handle_unknown="ignore"
        ), ["Color"])
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

transformed = preprocessor.fit_transform(df)

columns = preprocessor.get_feature_names_out()

result = pd.DataFrame(transformed, columns=columns)

print("Transformed Data:")
print(result)


Transformed Data:
    Size  Satisfaction  Color_Blue  Color_Green  Color_Red  Order_Value
0    0.0           0.0         0.0          0.0        1.0        500.0
1    1.0           2.0         1.0          0.0        0.0        900.0
2    2.0           1.0         0.0          1.0        0.0       1500.0
3    1.0           3.0         0.0          0.0        1.0       1100.0
4    0.0           1.0         0.0          1.0        0.0        650.0
5    2.0           2.0         1.0          0.0        0.0       1700.0
6    3.0           3.0         0.0          0.0        1.0       2200.0
7    1.0           0.0         0.0          1.0        0.0        700.0
8    3.0           2.0         1.0          0.0        0.0       2100.0
9    0.0           1.0         0.0          0.0        1.0        550.0
10   2.0           3.0         1.0          0.0        0.0       1600.0
11   3.0           2.0         0.0          1.0        0.0       2300.0
